In [6]:
import sqlite3
import pandas as pd
import sys
import os

# Fix 1 — point to project folder
project_path = r'C:\Users\VISHNU\Downloads\nifty100_project'
sys.path.append(project_path)

# Fix 2 — set working directory so file paths work correctly
os.chdir(project_path)

from src.etl.loader import load_all_data

print("Libraries loaded!")
print("Working directory:", os.getcwd())

Libraries loaded!
Working directory: C:\Users\VISHNU\Downloads\nifty100_project


In [7]:
# Load all 11 clean datasets using our loader.py from Step 2
data = load_all_data()

print("Data loaded and ready for database!")

Loading all datasets...

Dataset Summary:
  Dataset                Rows   Cols
  -----------------------------------
  profitandloss          1164     15
  balancesheet           1165     13
  cashflow               1152      7
  companies                92     12
  analysis                 20      6
  documents              1585      4
  prosandcons              16      4
  sectors                  92      6
  market_cap              552      9
  financial_ratios       1184     16
  peer_groups              56      4

All datasets loaded and cleaned successfully!
Data loaded and ready for database!


In [8]:
# This creates nifty100.db if it doesn't exist
conn = sqlite3.connect('data/nifty100.db')

print("Connected to database!")
print("Database file created at: data/nifty100.db")

Connected to database!
Database file created at: data/nifty100.db


In [9]:
# Write each DataFrame as a table in the database
tables = {
    'profitandloss':    data['profitandloss'],
    'balancesheet':     data['balancesheet'],
    'cashflow':         data['cashflow'],
    'companies':        data['companies'],
    'analysis':         data['analysis'],
    'documents':        data['documents'],
    'prosandcons':      data['prosandcons'],
    'sectors':          data['sectors'],
    'market_cap':       data['market_cap'],
    'financial_ratios': data['financial_ratios'],
    'peer_groups':      data['peer_groups'],
}

for table_name, df in tables.items():
    df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"  Loaded → {table_name:20s} ({len(df)} rows)")

print("\nAll tables loaded into database!")

  Loaded → profitandloss        (1164 rows)
  Loaded → balancesheet         (1165 rows)
  Loaded → cashflow             (1152 rows)
  Loaded → companies            (92 rows)
  Loaded → analysis             (20 rows)
  Loaded → documents            (1585 rows)
  Loaded → prosandcons          (16 rows)
  Loaded → sectors              (92 rows)
  Loaded → market_cap           (552 rows)
  Loaded → financial_ratios     (1184 rows)
  Loaded → peer_groups          (56 rows)

All tables loaded into database!


In [10]:
# Query the database to confirm all tables are there
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables_in_db = cursor.fetchall()

print("Tables in nifty100.db:")
for table in tables_in_db:
    print(f"  ✅ {table[0]}")

Tables in nifty100.db:
  ✅ profitandloss
  ✅ balancesheet
  ✅ cashflow
  ✅ companies
  ✅ analysis
  ✅ documents
  ✅ prosandcons
  ✅ sectors
  ✅ market_cap
  ✅ financial_ratios
  ✅ peer_groups


In [11]:
# Let's query the database like a real analyst
# Get top 5 companies by sales in their latest year
query = """
    SELECT company_id, year, sales, net_profit
    FROM profitandloss
    WHERE year = '2024-03'
    ORDER BY sales DESC
    LIMIT 5
"""

result = pd.read_sql_query(query, conn)
print("Top 5 companies by Sales (FY2024):")
result

Top 5 companies by Sales (FY2024):


,company_id,year,sales,net_profit
0,RELIANCE,2024-03,899041,79020
1,LICI,2024-03,845966,40916
2,IOC,2024-03,776352,43161
3,ONGC,2024-03,591396,57101
4,BPCL,2024-03,448083,26859


In [12]:
# This creates load_audit.csv — a record of how many rows
# were loaded into each table (required project deliverable!)
audit_rows = []

for table_name in tables.keys():
    cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
    row_count = cursor.fetchone()[0]
    audit_rows.append({
        'table':     table_name,
        'rows':      row_count,
        'status':    'OK' if row_count > 0 else 'EMPTY'
    })

audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv('output/load_audit.csv', index=False)

print("load_audit.csv saved to output/!")
print()
print(audit_df.to_string(index=False))

load_audit.csv saved to output/!

           table  rows status
   profitandloss  1164     OK
    balancesheet  1165     OK
        cashflow  1152     OK
       companies    92     OK
        analysis    20     OK
       documents  1585     OK
     prosandcons    16     OK
         sectors    92     OK
      market_cap   552     OK
financial_ratios  1184     OK
     peer_groups    56     OK


In [13]:
conn.close()
print("Database connection closed.")
print("\nStep 3 Complete!")
print("Your database is saved at: data/nifty100.db")

Database connection closed.

Step 3 Complete!
Your database is saved at: data/nifty100.db
